# ML Pipelines


1. Build an ML pipeline from scratch that chains imputation, scaling, encoding, and model training into a single reproducible object
2. Identify data leakage scenarios and explain how pipelines prevent them by fitting transformers only on training data
3. Construct a ColumnTransformer that applies different preprocessing to numeric and categorical features
4. Implement pipeline serialization and demonstrate that the same fitted pipeline produces identical results in training and production

**A pipeline is an ordered sequence of data transformations followed by a model. Each step takes the output of the previous step as input. The entire pipeline is fitted once on training data. At inference time, the same fitted pipeline transforms new data and produces predictions.**



In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression()),
])

# pipe.fit(X_train, y_train)
# predictions = pipe.predict(X_test)

In [ ]:
from sklearn.datasets import load_iris
iris = load_iris()

In [ ]:
X=iris["data"]
y=iris["target"]

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

In [ ]:
pipe = Pipeline([
    ("scaler",StandardScaler()),
    ("model",LogisticRegression()),
])

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)


In [ ]:
pipe.fit(X_train,y_train)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import GradientBoostingClassifier

numeric_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])

categorical_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("encode", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipe, ["age", "income", "score"]),
    ("cat", categorical_pipe, ["city", "gender", "plan"]),
])

full_pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", GradientBoostingClassifier()),
])

In [ ]:
# full_pipeline.fit(X_train,y_train)

In [ ]:
import mlflow

with mlflow.start_run():
    mlflow.log_param("max_depth", 5)
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("learning_rate", 0.1)

    pipe.fit(X_train, y_train)
    accuracy = pipe.score(X_test, y_test)

    mlflow.log_metric("accuracy", accuracy)
    mlflow.sklearn.log_model(pipe, "model")

In [ ]:
import wandb

wandb.init(project="my-pipeline")
wandb.config.update({"max_depth": 5, "n_estimators": 100})

pipe.fit(X_train, y_train)
accuracy = pipe.score(X_test, y_test)

wandb.log({"accuracy": accuracy})

### The typical progression:

1. Notebook exploration: Quick experiments, visualizations, feature ideas
2. Extract functions: Move preprocessing, feature engineering, evaluation into modules
3. Build Pipeline: Chain transformations into a sklearn Pipeline or custom class
4. Config management: Move all hyperparameters into a YAML/JSON config
5. Experiment tracking: Add MLflow or wandb logging
6. Data validation: Check schema, distributions, and missing value patterns before training
7. Tests: Unit tests for transformers, integration tests for the full pipeline
8. Deployment: Serialize the pipeline, wrap in an API (FastAPI, Flask), containerize
